Instructions listed here: https://docs.google.com/document/d/1fVrqrTK8T-jsPx8hfxyVfxf6P1quIQnM/edit

In [1]:
# run this to automatically reload modules when they are changed
%load_ext autoreload
%autoreload 2

### Setup

In [2]:
from Common_Functions import *
import numpy as np
from datetime import datetime, timedelta

/Users/jenny.wang2/anaconda3/envs/my_env/lib/python3.11/site-packages/snowflake/connector/vendored/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/Users/jenny.wang2/anaconda3/envs/my_env/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Python Key File


In [3]:
mydb_conn = mydb_conn_func()
email = 'jallen@teampurpose.com'
sub_advertiser_id = [46398, 47692]
sub_advertiser_id_string = ",".join(str(x) for x in sub_advertiser_id)
sub_advertiser_id_string

'46398,47692'

##### nativead pull

In [30]:
mysql_query = f"""
    SELECT  campaigns.id as campaign_id,
            campaigns.name as campaign_name,
            campaigns.type AS channel_type,
            CASE
                WHEN campaigns.subtype = 0 THEN 'Standard'
                WHEN campaigns.subtype = 1 THEN 'Blended In-Game'
                WHEN campaigns.subtype = 2 THEN 'OTT'
                WHEN campaigns.subtype IN (3,4) THEN 'DOOH'
                WHEN campaigns.subtype IN (5,6) THEN 'App-install'
                WHEN campaigns.subtype = 7 THEN 'Cable Video-On-Demand'
                WHEN campaigns.subtype = 8 THEN 'Live Linear TV'
                ELSE campaigns.subtype
            END AS channel_subtype,
            campaigns.start_date,
            campaigns.end_date,
            users.id as user_id,
            users.company_name as agency_name,
            users.account_name as account_name,
            users.timezone,
            users.team_admin_email as email,
            campaigns.sub_advertiser_id,
            sub_advertisers.name as sub_advertiser_name,
            campaigns_conversion_trackers.id as conversion_tracker_id,
            campaigns_conversion_trackers.tracker_type,
            conversion_trackers.name as conversion_tracker_name,
            conversion_trackers.unique_key as conversion_tracker_unique_key
    FROM    campaigns
            LEFT JOIN users
                ON campaigns.user_id = users.id
            LEFT JOIN sub_advertisers
                ON campaigns.sub_advertiser_id = sub_advertisers.id
            LEFT JOIN campaigns_conversion_trackers
                ON campaigns.id = campaigns_conversion_trackers.campaign_id
            LEFT JOIN conversion_trackers
                ON campaigns_conversion_trackers.conversion_tracker_id = conversion_trackers.id
    WHERE   campaigns.sub_advertiser_id IN ({sub_advertiser_id_string})
            and users.team_admin_email = 'jallen@teampurpose.com'
"""
print(mysql_query)


    SELECT  campaigns.id as campaign_id,
            campaigns.name as campaign_name,
            campaigns.type AS channel_type,
            CASE
                WHEN campaigns.subtype = 0 THEN 'Standard'
                WHEN campaigns.subtype = 1 THEN 'Blended In-Game'
                WHEN campaigns.subtype = 2 THEN 'OTT'
                WHEN campaigns.subtype IN (3,4) THEN 'DOOH'
                WHEN campaigns.subtype IN (5,6) THEN 'App-install'
                WHEN campaigns.subtype = 7 THEN 'Cable Video-On-Demand'
                WHEN campaigns.subtype = 8 THEN 'Live Linear TV'
                ELSE campaigns.subtype
            END AS channel_subtype,
            campaigns.start_date,
            campaigns.end_date,
            users.id as user_id,
            users.company_name as agency_name,
            users.account_name as account_name,
            users.timezone,
            users.team_admin_email as email,
            campaigns.sub_advertiser_id,
            sub_advertiser

In [32]:
mysql_df = pd.read_sql(mysql_query, con = mydb_conn)
mysql_df.head(5)

,campaign_id,campaign_name,channel_type,channel_subtype,start_date,end_date,user_id,agency_name,account_name,timezone,email,sub_advertiser_id,sub_advertiser_name,conversion_tracker_id,tracker_type,conversion_tracker_name,conversion_tracker_unique_key
0,442406,Lower Funnel - Display - All States - Website ...,DisplayCampaign,Standard,2026-03-02 05:00:00,2026-04-01 03:59:59,23058,Purpose Financial,Purpose Financial,Eastern Time (US & Canada),jallen@teampurpose.com,47692,Advance America - 2023 - ADDED VALUE,797978.0,primary,"Primary - App Submission New Customer, Legacy",M4X4fhBnOVvCgq2XCJam3a
1,442406,Lower Funnel - Display - All States - Website ...,DisplayCampaign,Standard,2026-03-02 05:00:00,2026-04-01 03:59:59,23058,Purpose Financial,Purpose Financial,Eastern Time (US & Canada),jallen@teampurpose.com,47692,Advance America - 2023 - ADDED VALUE,797979.0,primary,"Primary - NC App Submission, PWA",RyzLLSPMLTTF14Gy3hQcXe
2,442406,Lower Funnel - Display - All States - Website ...,DisplayCampaign,Standard,2026-03-02 05:00:00,2026-04-01 03:59:59,23058,Purpose Financial,Purpose Financial,Eastern Time (US & Canada),jallen@teampurpose.com,47692,Advance America - 2023 - ADDED VALUE,797980.0,secondary,AdvanceAmerica.net,AZr4XZdrwJUW2sOciWugvD
3,442406,Lower Funnel - Display - All States - Website ...,DisplayCampaign,Standard,2026-03-02 05:00:00,2026-04-01 03:59:59,23058,Purpose Financial,Purpose Financial,Eastern Time (US & Canada),jallen@teampurpose.com,47692,Advance America - 2023 - ADDED VALUE,797981.0,secondary,"Secondary - Apply Now, Legacy & PWA",vr7v6iDeVQHElac6GJCkiT
4,442406,Lower Funnel - Display - All States - Website ...,DisplayCampaign,Standard,2026-03-02 05:00:00,2026-04-01 03:59:59,23058,Purpose Financial,Purpose Financial,Eastern Time (US & Canada),jallen@teampurpose.com,47692,Advance America - 2023 - ADDED VALUE,797982.0,secondary,Advance America - Find a Store,jRRznBstPIwVDijkRvLReo


In [8]:
user_id = mysql_df['user_id'][0].item()
user_id

23058

#### 1. List all active campaigns in the last two months broken out by channel.

In [33]:
active_campaigns = mysql_df[
    (pd.to_datetime(mysql_df['end_date']) >= datetime.now() - pd.DateOffset(months=2))
][['campaign_id', 'campaign_name', 'channel_type', 'channel_subtype', 'start_date', 'end_date']].sort_values('channel_type') 

active_campaigns

,campaign_id,campaign_name,channel_type,channel_subtype,start_date,end_date
4233,3148905,Advance America - Retention - CTV - All States...,CtvCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59
4234,3148905,Advance America - Retention - CTV - All States...,CtvCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59
4245,3148905,Advance America - Retention - CTV - All States...,CtvCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59
4243,3148905,Advance America - Retention - CTV - All States...,CtvCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59
4242,3148905,Advance America - Retention - CTV - All States...,CtvCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59
...,...,...,...,...,...,...
4222,3148896,Advance America - Retention - Video - All Stat...,VideoCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59
4221,3148896,Advance America - Retention - Video - All Stat...,VideoCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59
4229,3148896,Advance America - Retention - Video - All Stat...,VideoCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59
4225,3148896,Advance America - Retention - Video - All Stat...,VideoCampaign,Standard,2026-03-01 05:00:00,2026-04-01 03:59:59


In [34]:
channel_detail = (
    active_campaigns
    .groupby(["channel_type"], dropna=False)
    .agg(
        active_campaign_count=("campaign_id", "nunique"),
    )
    .reset_index()
    .sort_values(["active_campaign_count", "channel_type"], ascending=[False, True])
)

channel_detail

,channel_type,active_campaign_count
1,DisplayCampaign,104
2,NativeCampaign,103
0,CtvCampaign,1
3,VideoCampaign,1


#### 2. Provide a list of conversion trackers with their ID, name, tracker type, conversion category, and unique key under (Sub_Advertiser ID) 46398 & 47692.

In [35]:
mysql_df[['user_id', 'sub_advertiser_id', 'campaign_id', 'conversion_tracker_id', 'tracker_type']].drop_duplicates()

,user_id,sub_advertiser_id,campaign_id,conversion_tracker_id,tracker_type
0,23058,47692,442406,797978.0,primary
1,23058,47692,442406,797979.0,primary
2,23058,47692,442406,797980.0,secondary
3,23058,47692,442406,797981.0,secondary
4,23058,47692,442406,797982.0,secondary
...,...,...,...,...,...
5653,23058,46398,3200453,2486969.0,secondary
5654,23058,46398,3200453,2486970.0,secondary
5655,23058,46398,3200453,2486971.0,secondary
5656,23058,46398,3200453,2486972.0,secondary


##### Redshift Data

In [21]:
#get the user_id
user_id = mysql_df['user_id'].unique()[0].item()
user_id

23058

In [48]:
# convert times from utc to eastern
[start_date_utc, end_date_utc] = utc_translation('2024-12-01 00:00:00', 
                                                 '2024-12-31 00:00:00', 
                                                 'US/Eastern')
end_date_utc

'2024-12-31 05:00:00'

#### 3. Provide the following KPIs for all display campaigns in December 2024:
- Impressions
- Clicks                                       
- Media Spend (LC)
- CPM
- CTR
- Engagements
- Engagement Rate

In [52]:
# redshift query pull
select_stmt = """
    select  advertiser_id,
            campaign_id,
            sum(has_won) as impressions,
            sum(has_click) as clicks,
            sum(has_conv) as conversions,
            sum(has_engagement) as engagements,
            sum(cost)/1000000.0 as media_spend
""" 

where_stmt = """
    where   sub_advertiser_id in (46398, 47692)
            and supply_inventory_type in ('display')
"""

groupby_stmt = "group by advertiser_id, campaign_id"

orderby_stmt = "order by campaign_id"

In [53]:
rs_df = daily_looper_fun(dur = 30,
                        enddate = '2024-12-31 05:00:00',
                        SELECT = select_stmt,
                        WHERE_daily = where_stmt,
                        GROUPBY = groupby_stmt,
                        ORDERBY = orderby_stmt,
                        user_id = user_id)
rs_df.head(10)

Mod Value : 2
Daily Cluster: 2
Mod Value : 2
Daily Cluster: 2
Pulling daily tables from daily DB
sa_rs_evt_table_2025_01_01
sa_rs_evt_table_2024_12_31
sa_rs_evt_table_2024_12_30
sa_rs_evt_table_2024_12_29
sa_rs_evt_table_2024_12_28
sa_rs_evt_table_2024_12_27
sa_rs_evt_table_2024_12_26
sa_rs_evt_table_2024_12_25
sa_rs_evt_table_2024_12_24
sa_rs_evt_table_2024_12_23
sa_rs_evt_table_2024_12_22
sa_rs_evt_table_2024_12_21
sa_rs_evt_table_2024_12_20
sa_rs_evt_table_2024_12_19
sa_rs_evt_table_2024_12_18
sa_rs_evt_table_2024_12_17
sa_rs_evt_table_2024_12_16
sa_rs_evt_table_2024_12_15
sa_rs_evt_table_2024_12_14
sa_rs_evt_table_2024_12_13
sa_rs_evt_table_2024_12_12
sa_rs_evt_table_2024_12_11
sa_rs_evt_table_2024_12_10
sa_rs_evt_table_2024_12_09
sa_rs_evt_table_2024_12_08
sa_rs_evt_table_2024_12_07
sa_rs_evt_table_2024_12_06
sa_rs_evt_table_2024_12_05
sa_rs_evt_table_2024_12_04
sa_rs_evt_table_2024_12_03
sa_rs_evt_table_2024_12_02


,advertiser_id,campaign_id,impressions,clicks,conversions,engagements,media_spend
0,23058,413265,0,0,4,0,0.000000
1,23058,413266,0,0,1,0,0.000000
2,23058,413267,44658,390,72,39,218.908096
3,23058,415741,138559,1859,134,176,599.713951
4,23058,415745,77172,314,98,36,407.014785
5,23058,415746,60335,257,47,32,312.361202
6,23058,439073,0,0,0,0,0.000000
7,23058,439075,0,0,0,0,0.000000
8,23058,439076,10164,171,9,13,48.517503
9,23058,439104,39915,343,24,34,178.227661


In [54]:
rs_df['click_through_rate'] = rs_df['clicks'] / rs_df['impressions'].replace(0, np.nan)
rs_df['conversion_rate'] = rs_df['conversions'] / rs_df['impressions'].replace(0, np.nan)
rs_df['engagement_rate'] = rs_df['engagements'] / rs_df['impressions'].replace(0, np.nan)
rs_df['cost_per_click'] = rs_df['media_spend'] / rs_df['clicks'].replace(0, np.nan)

In [55]:
rs_df[['advertiser_id', 'campaign_id', 'impressions', 'clicks', 'conversions', 'media_spend', 'click_through_rate', 'conversion_rate', 'engagements', 'engagement_rate', 'cost_per_click']].head(10)

,advertiser_id,campaign_id,impressions,clicks,conversions,media_spend,click_through_rate,conversion_rate,engagements,engagement_rate,cost_per_click
0,23058,413265,0,0,4,0.000000,NaN,NaN,0,NaN,NaN
1,23058,413266,0,0,1,0.000000,NaN,NaN,0,NaN,NaN
2,23058,413267,44658,390,72,218.908096,0.008733,0.001612,39,0.000873,0.561303
3,23058,415741,138559,1859,134,599.713951,0.013417,0.000967,176,0.001270,0.322600
4,23058,415745,77172,314,98,407.014785,0.004069,0.001270,36,0.000466,1.296225
5,23058,415746,60335,257,47,312.361202,0.004260,0.000779,32,0.000530,1.215413
6,23058,439073,0,0,0,0.000000,NaN,NaN,0,NaN,NaN
7,23058,439075,0,0,0,0.000000,NaN,NaN,0,NaN,NaN
8,23058,439076,10164,171,9,48.517503,0.016824,0.000885,13,0.001279,0.283728
9,23058,439104,39915,343,24,178.227661,0.008593,0.000601,34,0.000852,0.519614


#### 4. Provide KPIs for the CTV & Video campaigns active on Feb 1, 2025.

In [62]:
# redshift query pull
select_stmt = """
    select  advertiser_id,
            campaign_id,
            sum(has_won) as impressions,
            sum(has_click) as clicks,
            sum(has_conv) as conversions,
            sum(has_engagement) as engagements,
            sum(cost)/1000000.0 as media_spend,
            sum(cost_usd)/1000000.0 as media_spend_usd
""" 

where_stmt = """
    where   sub_advertiser_id in (46398, 47692)
            and supply_inventory_type in ('image', 'video')
"""

groupby_stmt = "group by advertiser_id, campaign_id"

orderby_stmt = "order by campaign_id"

In [63]:
# convert times from utc to eastern
[start_date_utc, end_date_utc] = utc_translation('2025-02-01 00:00:00', 
                                                 '2025-02-01 00:00:00', 
                                                 'US/Eastern')
end_date_utc

'2025-02-01 05:00:00'

In [64]:
rs_df2 = daily_looper_fun(dur = 1,
                        enddate = '2025-02-01 05:00:00',
                        SELECT = select_stmt,
                        WHERE_daily = where_stmt,
                        GROUPBY = groupby_stmt,
                        ORDERBY = orderby_stmt,
                        user_id = user_id)
rs_df2

Mod Value : 2
Daily Cluster: 2
Mod Value : 2
Daily Cluster: 2
Pulling daily tables from daily DB
sa_rs_evt_table_2025_02_02
sa_rs_evt_table_2025_02_01


,advertiser_id,campaign_id,impressions,clicks,conversions,engagements,media_spend,media_spend_usd
0,23058,413261,14285,66,36,6,64.533901,64.533901
1,23058,415754,70618,163,137,26,154.987024,154.987024
2,23058,415763,19468,31,66,7,97.135675,97.135675
3,23058,415764,3006,6,17,1,14.361657,14.361657
4,23058,439058,3729,21,5,1,16.847270,16.847270
...,...,...,...,...,...,...,...,...
71,23058,1631791,1281,2,14,1,3.475806,3.475806
72,23058,1637409,28994,56,53,9,72.495208,72.495208
73,23058,1643759,9251,27,12,6,26.482712,26.482712
74,23058,1643857,0,0,2,0,0.000000,0.000000


In [65]:
rs_df2['click_through_rate'] = rs_df2['clicks'] / rs_df2['impressions'].replace(0, np.nan)
rs_df2['conversion_rate'] = rs_df2['conversions'] / rs_df2['impressions'].replace(0, np.nan)
rs_df2['engagement_rate'] = rs_df2['engagements'] / rs_df2['impressions'].replace(0, np.nan)
rs_df2['cost_per_click'] = rs_df2['media_spend'] / rs_df2['clicks'].replace(0, np.nan)

In [61]:
rs_df2[['advertiser_id', 'campaign_id', 'impressions', 'clicks', 'conversions', 'media_spend', 'click_through_rate', 'conversion_rate', 'engagements', 'engagement_rate', 'cost_per_click']].head(10)

,advertiser_id,campaign_id,impressions,clicks,conversions,media_spend,click_through_rate,conversion_rate,engagements,engagement_rate,cost_per_click
0,23058,413261,14285,66,36,64.533901,0.004620,0.002520,6,0.000420,0.977786
1,23058,415754,70618,163,137,154.987024,0.002308,0.001940,26,0.000368,0.950841
2,23058,415763,19468,31,66,97.135675,0.001592,0.003390,7,0.000360,3.133409
3,23058,415764,3006,6,17,14.361657,0.001996,0.005655,1,0.000333,2.393609
4,23058,439058,3729,21,5,16.847270,0.005632,0.001341,1,0.000268,0.802251
5,23058,439106,12028,23,34,46.409022,0.001912,0.002827,2,0.000166,2.017784
6,23058,439118,6798,7,21,35.814010,0.001030,0.003089,1,0.000147,5.116287
7,23058,439119,1619,3,7,7.759992,0.001853,0.004324,0,0.000000,2.586664
8,23058,442413,280,3,4,1.009381,0.010714,0.014286,1,0.003571,0.336460
9,23058,442421,105,1,18,0.547926,0.009524,0.171429,1,0.009524,0.547926


#### 5. Please provide a breakdown of Cost KPIs (cost, media cost in USD, CPM, CPC) by region and city for one campaign in the last 5 days.

In [77]:
# redshift query pull
select_stmt = """
    select  region,
            city,
            sum(has_won) as impressions,
            sum(has_click) as clicks,
            sum(cost)/1000000.0 as media_spend,
            sum(cost_usd)/1000000.0 as media_spend_usd
""" 

where_stmt = """
    where   campaign_id in (2733954)
"""

groupby_stmt = "group by region, city"

orderby_stmt = "order by region, city"

In [78]:
rs_df3 = daily_looper_fun(dur = 1,
                        enddate = '2026-03-30 05:00:00',
                        SELECT = select_stmt,
                        WHERE_daily = where_stmt,
                        GROUPBY = groupby_stmt,
                        ORDERBY = orderby_stmt,
                        user_id = user_id)
rs_df3

Mod Value : 2
Daily Cluster: 2
Mod Value : 2
Daily Cluster: 2
Pulling daily tables from daily DB
sa_rs_evt_table_2026_03_31
sa_rs_evt_table_2026_03_30


,region,city,impressions,clicks,media_spend,media_spend_usd
0,ab,calgary,38163,30,64.162748,64.162748
1,bc,vancouver,17930,16,36.112480,36.112480
2,on,mississauga,27104,19,57.984362,57.984362
3,on,ottawa,22755,20,52.076566,52.076566
4,ab,calgary,85355,64,111.867019,111.867019
5,bc,vancouver,33618,23,45.042891,45.042891
6,on,mississauga,40817,35,56.275055,56.275055
7,on,ottawa,30889,21,42.298061,42.298061


In [79]:
rs_df3['cost_per_click'] = rs_df3['media_spend'] / rs_df3['clicks'].replace(0, np.nan)

In [80]:
rs_df3

,region,city,impressions,clicks,media_spend,media_spend_usd,cost_per_click
0,ab,calgary,38163,30,64.162748,64.162748,2.138758
1,bc,vancouver,17930,16,36.112480,36.112480,2.257030
2,on,mississauga,27104,19,57.984362,57.984362,3.051809
3,on,ottawa,22755,20,52.076566,52.076566,2.603828
4,ab,calgary,85355,64,111.867019,111.867019,1.747922
5,bc,vancouver,33618,23,45.042891,45.042891,1.958387
6,on,mississauga,40817,35,56.275055,56.275055,1.607859
7,on,ottawa,30889,21,42.298061,42.298061,2.014193
